### Structured Output

Models can be requested to provide their resposne in a format matching a given schema. This is useful for ensuring the output can be easily parsed adn used in a subsequent processing. LangChain supports mulitple schema types and methods for enforcing structured output.

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Pydantic

Pydantic models provide the riches feature set with field validation, descriptions and nested Structures

In [8]:
from langchain.chat_models import init_chat_model
model = init_chat_model("openai:gpt-4.1")
model


ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}}, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'mid_conversation_system_messages': True, 'mid_conversation_tools': False}, client=<openai.resources.chat.completions.completions.Completions object at 0x119882020>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x119ae6aa0

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    genre: str = Field(..., description="The genre of the movie")
    rating: float = Field(..., description="The rating of the movie on a scale of 0 to 10")

In [13]:
model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}}, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'mid_conversation_system_messages': True, 'mid_conversation_tools': False}, client=<openai.resources.chat.completions.completions.Completions object at 0x119882020>, async_client=<openai.resources.chat.completions.completions.AsyncCompletio

In [16]:
response = model.invoke("Tell me about the movie Inception.")
print(response.content)

**Inception** is a 2010 science fiction action film written and directed by Christopher Nolan. The movie explores themes of dreams, reality, and the power of the subconscious mind. 

**Plot Overview:**
The story centers on Dom Cobb (played by Leonardo DiCaprio), a skilled thief who specializes in "extraction"—the art of stealing secrets from deep within people's subconscious during their dreams. Cobb is given a challenging task: instead of extracting an idea, he must perform "inception"—planting an idea in someone’s mind so subtly that they believe it originated from their own thoughts. If he succeeds, Cobb may finally be able to return home to his family. However, the mission is complicated by his personal demons and the shifting, dangerous landscape of the dream world.

**Main Cast:**
- Leonardo DiCaprio as Dom Cobb
- Joseph Gordon-Levitt as Arthur
- Ellen Page (now Elliot Page) as Ariadne
- Tom Hardy as Eames
- Ken Watanabe as Saito
- Cillian Murphy as Robert Fischer
- Marion Cotill

In [14]:
model_with_structured_output.invoke("Tell me about the movie Inception.")

Movie(title='Inception', year=2010, director='Christopher Nolan', genre='Science Fiction, Action, Thriller', rating=8.8)

In [17]:
model_with_structured_output.invoke("Tell me about the movie 24.")

Movie(title='24', year=2016, director='Vikram Kumar', genre='Science Fiction, Thriller', rating=7.8)

#### Message output alongside parsed structure

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    genre: str = Field(..., description="The genre of the movie")
    rating: float = Field(..., description="The rating of the movie on a scale of 0 to 10")

# Shows both the structured output and the raw output from the model
model_with_structure = model.with_structured_output(Movie,include_raw=True)
response = model_with_structure.invoke("Tell me about the movie Inception.")
print(response)

{'raw': AIMessage(content='{"title":"Inception","year":2010,"director":"Christopher Nolan","genre":"Science Fiction, Action, Thriller","rating":8.8}', additional_kwargs={'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', genre='Science Fiction, Action, Thriller', rating=8.8), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 145, 'total_tokens': 176, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_36620df7ef', 'id': 'chatcmpl-EbLs2MSbIG6SErLhXlQ4bQvTcQnmA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a15d8d-728f-775d-b305-db76637c0b53-0', tool

### Nested Output

In [26]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    birth_year: int = Field(..., description="The birth year of the actor")
    nationality: str = Field(..., description="The nationality of the actor")
    awards: list[str] = Field(..., description="A list of awards won by the actor")

class MovieDetails(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    genre: str = Field(..., description="The genre of the movie")
    rating: float = Field(..., description="The rating of the movie on a scale of 0 to 10")
    cast: list[Actor] = Field(..., description="A list of actors in the movie")
    budget: float | None  = Field(..., description="The budget of the movie in  millions of USD")


model_with_nested_structure = model.with_structured_output(MovieDetails)
response = model_with_nested_structure.invoke("Tell me about the movie Inception.")
print(response)

title='Inception' year=2010 director='Christopher Nolan' genre='Science Fiction, Action, Thriller' rating=8.8 cast=[Actor(name='Leonardo DiCaprio', birth_year=1974, nationality='American', awards=['Academy Award for Best Actor (2016)', 'Golden Globe Awards']), Actor(name='Joseph Gordon-Levitt', birth_year=1981, nationality='American', awards=['Primetime Emmy Award (as producer)']), Actor(name='Elliot Page', birth_year=1987, nationality='Canadian', awards=['Academy Award Nomination']), Actor(name='Tom Hardy', birth_year=1977, nationality='British', awards=['BAFTA Rising Star Award']), Actor(name='Ken Watanabe', birth_year=1959, nationality='Japanese', awards=['Japan Academy Prize', 'Academy Award Nomination'])] budget=160.0


### TypeDict

TypeDict provides a simpler alternative using python's built-in typing, ideal when you don't need runtime validation

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A Movie dictionary with details about a movie."""
    title: Annotated[str, ..., "The title of the movie"] # The ..., indicates that this field is required
    year: Annotated[int, "The release year of the movie"]
    director: Annotated[str, "The director of the movie"]
    genre: Annotated[str, "The genre of the movie"]
    rating: Annotated[float, "The rating of the movie on a scale of 0 to 10"]
    cast: Annotated[list[dict], "A list of actors in the movie"]
    budget: Annotated[float | None, "The budget of the movie in millions of USD"]


model_with_typed_dict = model.with_structured_output(MovieDict)
response = model_with_typed_dict.invoke("Tell me about the movie Avengers.")
print(response) 

{'title': 'The Avengers', 'year': 2012, 'director': 'Joss Whedon', 'genre': 'Action, Adventure, Sci-Fi', 'rating': 8.0, 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark/Iron Man'}, {'name': 'Chris Evans', 'role': 'Steve Rogers/Captain America'}, {'name': 'Mark Ruffalo', 'role': 'Bruce Banner/Hulk'}, {'name': 'Chris Hemsworth', 'role': 'Thor'}, {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff/Black Widow'}, {'name': 'Jeremy Renner', 'role': 'Clint Barton/Hawkeye'}, {'name': 'Tom Hiddleston', 'role': 'Loki'}, {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}], 'budget': 220000000}


#### Nested Structure

In [34]:


class Actor(TypedDict):
    name: str 
    birth_year: int 
    nationality: str 
    awards: list[str] 

class MovieDetails(BaseModel):
    title: str 
    year: int 
    director: str 
    genre: str 
    rating: float 
    cast: list[Actor] 
    budget: float | None  


model_with_nested_structure = model.with_structured_output(MovieDetails)
response = model_with_nested_structure.invoke("Tell me about the movie Avengers.")
print(response)

title='The Avengers' year=2012 director='Joss Whedon' genre='Action, Adventure, Sci-Fi' rating=8.0 cast=[{'name': 'Robert Downey Jr.', 'birth_year': 1965, 'nationality': 'American', 'awards': ['Academy Award Nominee', 'Golden Globe Winner']}, {'name': 'Chris Evans', 'birth_year': 1981, 'nationality': 'American', 'awards': ["People's Choice Award Winner"]}, {'name': 'Mark Ruffalo', 'birth_year': 1967, 'nationality': 'American', 'awards': ['Primetime Emmy Award Winner']}, {'name': 'Chris Hemsworth', 'birth_year': 1983, 'nationality': 'Australian', 'awards': ["People's Choice Award Winner"]}, {'name': 'Scarlett Johansson', 'birth_year': 1984, 'nationality': 'American', 'awards': ['Tony Award Winner', 'BAFTA Winner']}, {'name': 'Jeremy Renner', 'birth_year': 1971, 'nationality': 'American', 'awards': ['Academy Award Nominee']}, {'name': 'Tom Hiddleston', 'birth_year': 1981, 'nationality': 'British', 'awards': ['Laurence Olivier Award Winner']}] budget=220000000.0


In [ ]:
model_with_typed_dict.profile # Cannot see the profile for TypedDict models yet, but it will be available in a future release of LangChain.

AttributeError: 'RunnableSequence' object has no attribute 'profile'

In [36]:
model.profile

{'name': 'GPT-4.1',
 'release_date': '2025-04-14',
 'last_updated': '2025-04-14',
 'open_weights': False,
 'max_input_tokens': 1047576,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'pdf_inputs': True,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True,
 'tool_call_streaming': True,
 'mid_conversation_system_messages': True,
 'mid_conversation_tools': False}

In [ ]:
import os
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email address of the contact")
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(
    model = "gpt-5",
    response_format = ContactInfo,  # Autoselects the correct response format based on the model's capabilities
    )

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact information from the following text: 'John Doe, john.doe@example.com,123-456-7890'"}]
})

print(result,end="\n")  # This will print the raw output from the model
print(result["structured_response"])  # This will print the structured output as a ContactInfo object

{'messages': [HumanMessage(content="Extract contact information from the following text: 'John Doe, john.doe@example.com,123-456-7890'", additional_kwargs={}, response_metadata={}, id='d6d6644b-b9ed-4689-bd4e-db8f5a9689a2'), AIMessage(content='{"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 996, 'prompt_tokens': 202, 'total_tokens': 1198, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 960, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EbN0TB9Ni6iuqXO4PRouJtrGNCBGx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--0

In [44]:
import os
from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model = "gpt-5",
    response_format = ContactInfo,  # Autoselects the correct response format based on the model's capabilities
    )

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact information from the following text: 'John Doe, john.doe@example.com,123-456-7890'"}]
})

agent = create_agent(
    model = "gpt-5",
    response_format = ContactInfo,  # Autoselects the correct response format based on the model's capabilities
    )

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact information from the following text: 'John Doe, john.doe@example.com,123-456-7890'"}]
})

print(result,end="\n")  # This will print the raw output from the model
print(result["structured_response"])  # This will print the structured output as a ContactInfo object

{'messages': [HumanMessage(content="Extract contact information from the following text: 'John Doe, john.doe@example.com,123-456-7890'", additional_kwargs={}, response_metadata={}, id='9122bb3c-2f1d-46d3-a7e6-0b850984922d'), AIMessage(content='{"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1699, 'prompt_tokens': 183, 'total_tokens': 1882, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1664, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EbN3BKcVgx7xoSx1fXWcIZKpmWvHv', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run-

### DataClasses

A data class is a class typically containing mainly data, although they aren't really any restrictions. You create is using the @dataclass decorator 

In [45]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name: str # Name of the contact
    email: str  # Email address of the contact
    phone: str # Phone number of the contact

agent = create_agent(
    model = "gpt-5",
    response_format = ContactInfo,  # Autoselects the correct response format based on the model's capabilities
    )
result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact information from the following text: 'John Doe, john.doe@example.com,123-456-7890'"}]
})  
print(result,end="\n")  # This will print the raw output from the model
print(result["structured_response"])  # This will print the structured output as a ContactInfo object

{'messages': [HumanMessage(content="Extract contact information from the following text: 'John Doe, john.doe@example.com,123-456-7890'", additional_kwargs={}, response_metadata={}, id='eefd34a2-4e8a-4246-8592-c8dca67737a9'), AIMessage(content='{"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 228, 'prompt_tokens': 183, 'total_tokens': 411, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EbN5lGti4R9U0cbhqpXkgjLEYxAbk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01